In [ ]:
import os
import re
import threading
import requests
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# Configuration
# This folder matches DockerPreprocessing.ipynb's station/year input layout.
# ============================================================
base_url = "https://gage-data.earthscope.org/archive/gnss/rinex/obs/"
log_base_url = "https://gage-data.earthscope.org/archive/gnss/logs/"

token = os.environ["EARTHSCOPE_TOKEN"]
headers = {"Authorization": f"Bearer {token}"}

output_dir = "/Volumes/cornice/marquardt/data/GNSS/docker"

# Set this to your token's *real* expiration time
token_expiration = datetime.strptime("2026-12-31 23:59:59", "%Y-%m-%d %H:%M:%S")

start_year = 2010
end_year = 2026

MAX_WORKERS = 3

# Restart/skip behavior
SKIP_IF_EXISTS = True          # if True, won't re-download existing files
LOG_SKIPS = True              # if True, prints SKIP lines too (can be noisy)
VERIFY_SIZE_NONZERO = True     # if True, treats 0-byte files as broken and re-downloads

os.makedirs(output_dir, exist_ok=True)

# ============================================================
# The 29 paper stations (lowercase EarthScope station codes)
stations = [
    "gz01",
    "gz02",
    "gz06",
    "gz09",
    "gz13",
    "gz17",
    "gz18",
    "la01",
    "la02",
    "la03",
    "la04",
    "la05",
    "la06",
    "la07",
    "la08",
    "la09",
    "la10",
    "la11",
    "la12",
    "la13",
    "la14",
    "la15",
    "la16",
    "la17",
    "la18",
    "mg04",
    "mg05",
    "mg06",
    "mg07",
]

print(f"Using {len(stations)} hard-coded paper stations.", flush=True)

# Thread-safe printing
# ============================================================
print_lock = threading.Lock()

def ts_print(msg: str) -> None:
    with print_lock:
        print(msg, flush=True)

# ============================================================
# Helpers
# ============================================================
def is_leap_year(y: int) -> bool:
    return (y % 4 == 0) and ((y % 100 != 0) or (y % 400 == 0))

def days_in_year(y: int) -> int:
    return 366 if is_leap_year(y) else 365

def good_existing_file(path: str) -> bool:
    if not os.path.exists(path):
        return False
    if not VERIFY_SIZE_NONZERO:
        return True
    try:
        return os.path.getsize(path) > 0
    except OSError:
        return False

def download_log_file(session: requests.Session, station: str) -> str:
    url = f"{log_base_url}{station}log.txt"
    r = session.get(url, headers=headers, timeout=60)
    if r.status_code != 200:
        return "no_log_file"
    m = re.search(r"Monument Description\s*:\s*(.*)", r.text, re.IGNORECASE)
    return m.group(1).strip() if m else "unknown"

def download_rinex_day(session: requests.Session, station: str, year: int, doy: int) -> str:
    if datetime.now() > token_expiration:
        raise RuntimeError("Token has expired (based on token_expiration). Update token/token_expiration.")

    doy_str = f"{doy:03d}"
    yy = str(year)[-2:]
    fname = f"{station}{doy_str}0.{yy}d.Z"
    url = f"{base_url}{year}/{doy_str}/{fname}"

    station_dir = os.path.join(output_dir, station, str(year))
    os.makedirs(station_dir, exist_ok=True)
    out_path = os.path.join(station_dir, fname)

    # ✅ Restart support: skip if already downloaded
    if SKIP_IF_EXISTS and good_existing_file(out_path):
        return f"SKIP  {station} {year} {doy_str}  (exists)"

    r = session.get(url, headers=headers, timeout=120)

    if r.status_code == 200:
        with open(out_path, "wb") as f:
            f.write(r.content)

        # basic sanity check
        if VERIFY_SIZE_NONZERO and os.path.getsize(out_path) == 0:
            return f"BAD   {station} {year} {doy_str}  (0 bytes written)"

        return f"SAVED {station} {year} {doy_str}  -> {out_path}"

    return f"MISS  {station} {year} {doy_str}  (HTTP {r.status_code})"

def process_station(station: str) -> dict:
    results = {
        "station": station,
        "saved": 0,
        "missing": 0,
        "skipped": 0,
        "bad": 0,
        "errors": 0,
        "monument_description": None,
    }

    ts_print(f"[{station}] START")

    try:
        with requests.Session() as session:
            # monument description (optional)
            try:
                results["monument_description"] = download_log_file(session, station)
                ts_print(f"[{station}] LOG monument={results['monument_description']}")
            except Exception as e:
                results["monument_description"] = "error"
                results["errors"] += 1
                ts_print(f"[{station}] LOG ERROR {type(e).__name__}: {e}")

            for year in range(start_year, end_year + 1):
                for doy in range(1, days_in_year(year) + 1):
                    try:
                        msg = download_rinex_day(session, station, year, doy)

                        # per-file prints (you asked for this)
                        if msg.startswith("SKIP") and not LOG_SKIPS:
                            pass
                        else:
                            ts_print(msg)

                        if msg.startswith("SAVED"):
                            results["saved"] += 1
                        elif msg.startswith("MISS"):
                            results["missing"] += 1
                        elif msg.startswith("SKIP"):
                            results["skipped"] += 1
                        elif msg.startswith("BAD"):
                            results["bad"] += 1

                    except Exception as e:
                        results["errors"] += 1
                        ts_print(f"ERR   {station} {year} {doy:03d}  {type(e).__name__}: {e}")

    except Exception as e:
        results["errors"] += 1
        ts_print(f"[{station}] FATAL {type(e).__name__}: {e}")

    ts_print(
        f"[{station}] DONE saved={results['saved']} missing={results['missing']} "
        f"skipped={results['skipped']} bad={results['bad']} errors={results['errors']}"
    )
    return results

# ============================================================
# Run parallel by station
# ============================================================
summary_rows = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(process_station, st): st for st in stations}
    for fut in as_completed(futures):
        st = futures[fut]
        try:
            res = fut.result()
            summary_rows.append(res)
        except Exception as e:
            ts_print(f"[{st}] FAILED in future: {type(e).__name__}: {e}")

# ============================================================
# Write summaries
# ============================================================
summary_df = pd.DataFrame(summary_rows).sort_values("station")
summary_path = os.path.join(output_dir, "gnss_download_summary_by_station.csv")
summary_df.to_csv(summary_path, index=False)
ts_print(f"Wrote summary: {summary_path}")

